# Compare benchmark runs

Set `old` and `new` below, then run all cells. Paths work from either the repository root or `benchmarks/`. Requires **pandas**, plus the plotting/Jupyter packages in `notebook-requirements.txt`.

Only **shared valid cases** are plotted: implementation, workload, producer/consumer counts, capacity, and payload must match. Repeats are independent samples, not paired by repeat number; all valid repeats contribute to each run's median and minimum–maximum range. Coverage and failures remain visible even when a case cannot be compared.

Throughput counts **unique events**, not broadcast deliveries. Latency is scheduled-arrival-to-consumer time, including backpressure and queueing. Equal offered-load fractions can represent **different absolute rates** in different runs; the latency section shows those rates explicitly. These are local run comparisons, not universal performance rankings.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.lines import Line2D

old = "results/run-20261009T121858355119Z/"
new = "results/run-20261009T235317299333Z/"

BENCHMARKS_DIR = Path.cwd() if (Path.cwd() / "run.py").exists() else Path.cwd() / "benchmarks"
RUN_DIRS = {"old": Path(old), "new": Path(new)}
RUN_DIRS = {
    name: (path if path.is_absolute() else BENCHMARKS_DIR / path).resolve()
    for name, path in RUN_DIRS.items()
}
RUNS = list(RUN_DIRS)
COLORS = {"zigrupt": "#005A91", "java": "#B94A00", "rust": "#007A57"}
MARKERS = {"zigrupt": "o", "java": "s", "rust": "^"}
LINESTYLES = {"old": "--", "new": "-"}
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})

CASE_KEYS = ["workload", "implementation", "producers", "consumer_count", "capacity", "payload"]
LATENCY_KEYS = CASE_KEYS + ["fraction"]
PERCENTILES = ["p50_ns", "p99_ns", "p999_ns"]


def read_results(directory: Path, filename: str) -> pd.DataFrame:
    path = directory / filename
    if not path.exists() and filename == "latency.csv":
        return pd.DataFrame(columns=CASE_KEYS + ["mode", "repeat", "valid", "fraction"])
    frame = pd.read_csv(path)
    required = CASE_KEYS + ["mode", "repeat", "valid"]
    if filename == "latency.csv":
        required += ["fraction", "consumer", "samples", *PERCENTILES]
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    frame["valid"] = frame["valid"].astype(str).str.lower().eq("true")
    numeric = [
        "producers", "consumer_count", "capacity", "payload", "repeat", "fraction",
        "consumer", "samples", "offered_rate", "events_per_second", "cpu_cores",
        "peak_rss_bytes", *PERCENTILES,
    ]
    for field in numeric:
        if field in frame:
            frame[field] = pd.to_numeric(frame[field], errors="coerce").replace(
                [float("inf"), -float("inf")], float("nan")
            )
    # Optional counters remain missing, never zero-filled.
    for field in ["offered_rate", "events_per_second", "cpu_cores", "peak_rss_bytes", "cpus"]:
        if field not in frame:
            frame[field] = float("nan")
    return frame


throughput = {name: read_results(path, "throughput.csv") for name, path in RUN_DIRS.items()}
latency = {name: read_results(path, "latency.csv") for name, path in RUN_DIRS.items()}
otdf, ntdf = throughput["old"], throughput["new"]
oldf, nldf = latency["old"], latency["new"]
for name, path in RUN_DIRS.items():
    print(f"{name}: {path}")
    print(f"  {len(throughput[name])} throughput/check rows; {len(latency[name])} latency rows")
display(otdf.head(), ntdf.head(), oldf.head(), nldf.head())

## 1. Coverage and the intersection

Checks, calibration, blocked cases, and invalid measurements are not throughput samples. A latency repeat is usable only when every expected consumer appears exactly once, all rows are valid, samples are positive, and all three percentile counters are finite and nonnegative. A broadcast repeat contributes its **worst consumer** percentile, not one independent sample per consumer and not a pooled percentile.

In [ ]:
def valid_saturation(frame: pd.DataFrame) -> pd.DataFrame:
    selected = frame.loc[
        frame["mode"].eq("throughput") & frame["valid"]
        & frame["events_per_second"].gt(0)
        & frame[CASE_KEYS + ["repeat"]].notna().all(axis=1)
    ].copy()
    duplicate = selected.duplicated(CASE_KEYS + ["repeat"], keep=False)
    if duplicate.any():
        raise ValueError("Duplicate throughput repeats; do not count them as independent samples.")
    return selected


def complete_latency(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    accepted: list[dict[str, object]] = []
    rejected: list[dict[str, object]] = []
    keys = LATENCY_KEYS + ["repeat"]
    selected = frame.loc[frame["mode"].eq("latency")]
    for _, group in selected.groupby(keys, dropna=False, sort=True):
        first = group.iloc[0]
        expected = first["consumer_count"]
        ids = group["consumer"]
        complete = (
            group[keys].notna().all().all()
            and pd.notna(expected) and expected > 0 and float(expected).is_integer()
            and len(group) == expected and ids.notna().all()
            and sorted(ids.tolist()) == list(range(int(expected)))
            and group["valid"].all() and group["samples"].gt(0).all()
            and group[PERCENTILES].ge(0).all().all()
            and group["fraction"].gt(0).all() and group["fraction"].le(1).all()
            and group["offered_rate"].gt(0).all()
            and group["offered_rate"].nunique() == 1
        )
        record = {key: first[key] for key in keys}
        if not complete:
            rejected.append(record | {"issue": "Invalid, duplicate, or incomplete consumer repeat"})
            continue
        record.update({field: group[field].max() for field in PERCENTILES})
        # Rate and resource counters describe the process, so take them only once.
        record.update({field: first[field] for field in ["offered_rate", "events_per_second", "cpus"]})
        record["attainment_pct"] = first["events_per_second"] / first["offered_rate"] * 100
        accepted.append(record)
    columns = keys + PERCENTILES + ["offered_rate", "events_per_second", "cpus", "attainment_pct"]
    return pd.DataFrame(accepted, columns=columns), pd.DataFrame(rejected, columns=keys + ["issue"])


def intersect_cases(
    frames: dict[str, pd.DataFrame], keys: list[str]
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    common = frames[RUNS[0]][keys].drop_duplicates()
    for name in RUNS[1:]:
        common = common.merge(frames[name][keys].drop_duplicates(), on=keys, how="inner", validate="one_to_one")
    common = common.sort_values(keys).reset_index(drop=True)
    return common, {
        name: frame.merge(common, on=keys, how="inner", validate="many_to_one")
        for name, frame in frames.items()
    }


saturation = {name: valid_saturation(frame) for name, frame in throughput.items()}
latency_runs: dict[str, pd.DataFrame] = {}
incomplete: dict[str, pd.DataFrame] = {}
for name, frame in latency.items():
    latency_runs[name], incomplete[name] = complete_latency(frame)
shared_throughput, compared_throughput = intersect_cases(saturation, CASE_KEYS)
shared_latency, compared_latency = intersect_cases(latency_runs, LATENCY_KEYS)

coverage = []
for name in RUNS:
    t, l = throughput[name], latency[name]
    coverage.append({
        "run": name,
        "throughput attempts": int(t["mode"].eq("throughput").sum()),
        "valid throughput repeats": len(saturation[name]),
        "valid throughput cases": len(saturation[name][CASE_KEYS].drop_duplicates()),
        "shared throughput repeats": len(compared_throughput[name]),
        "complete latency repeats": len(latency_runs[name]),
        "rejected latency repeats": len(incomplete[name]),
        "shared latency repeats": len(compared_latency[name]),
    })
display(pd.DataFrame(coverage).set_index("run"))
print(f"Intersection: {len(shared_throughput)} throughput cases; {len(shared_latency)} latency case/load pairs.")
display(shared_throughput)

excluded = []
issues = []
for name in RUNS:
    for mode, frame, common, keys in [
        ("throughput", saturation[name], shared_throughput, CASE_KEYS),
        ("latency", latency_runs[name], shared_latency, LATENCY_KEYS),
    ]:
        cases = frame[keys].drop_duplicates().merge(common, on=keys, how="left", indicator=True)
        unmatched = cases.loc[cases["_merge"].eq("left_only"), keys]
        if not unmatched.empty:
            excluded.append(unmatched.assign(run=name, mode=mode))
    for mode, frame in [("throughput/check", throughput[name]), ("latency", latency[name])]:
        failed = frame.loc[frame["mode"].eq("blocked") | ~frame["valid"]].copy()
        if not failed.empty:
            fields = [field for field in CASE_KEYS + ["mode", "repeat", "fraction", "error", "reason"] if field in failed]
            issues.append(failed[fields].drop_duplicates().assign(run=name))
    if not incomplete[name].empty:
        issues.append(incomplete[name].assign(run=name, mode="latency"))
if excluded:
    display(Markdown("**Valid cases excluded because the other run has no matching valid case:**"))
    display(pd.concat(excluded, ignore_index=True))
if issues:
    display(Markdown("**Blocked, invalid, or incomplete attempts (not plotted):**"))
    display(pd.concat(issues, ignore_index=True).drop_duplicates())

## 2. Saturated throughput and resource costs

Implementation colors and markers match `visualize.ipynb`. **Old** uses an open marker; **new** a filled marker. Each marker is the median of all valid repeats; horizontal whiskers span minimum–maximum, **not a confidence interval**. CPU is CPU time / elapsed time. Peak RSS includes runtime overhead and uses a logarithmic axis. Resource counters are compared only where both runs report positive values.

In [ ]:
def summarize(
    frames: dict[str, pd.DataFrame], keys: list[str], field: str
) -> pd.DataFrame:
    summaries = []
    for name in RUNS:
        usable = frames[name].loc[frames[name][field].notna() & frames[name][field].gt(0)]
        summary = usable.groupby(keys)[field].agg(median="median", low="min", high="max", repeats="count").reset_index()
        summaries.append(summary.assign(run=name))
    result = pd.concat(summaries, ignore_index=True)
    # Optional counters may be absent even when the throughput case is shared.
    common = result.groupby(keys)["run"].nunique().reset_index()
    common = common.loc[common["run"].eq(len(RUNS)), keys]
    return result.merge(common, on=keys, how="inner", validate="many_to_one")


def legend_handles(implementations: list[str]) -> list[Line2D]:
    return [
        Line2D([0], [0], color=COLORS.get(impl, "#777777"), marker=MARKERS.get(impl, "D"),
               markerfacecolor="white" if name == "old" else COLORS.get(impl, "#777777"),
               markeredgecolor="#1A1A1A", markeredgewidth=0.8,
               linestyle=LINESTYLES[name], label=f"{impl} / {name}", lw=1.8, markersize=7)
        for impl in implementations for name in RUNS
    ]


def horizontal_panels(summary: pd.DataFrame, scale: float, unit: str, title: str, log_axis: bool = False) -> None:
    if summary.empty:
        display(Markdown(f"No shared valid measurements for {title.lower()}."))
        return
    implementations = sorted(summary["implementation"].unique())
    values = summary[["low", "high"]] / scale
    lower, upper = values["low"].min(), values["high"].max()
    limits = (lower / 1.7, upper * 1.7) if log_axis else (0, upper * 1.12)
    for (producers, consumers), selected in summary.groupby(["producers", "consumer_count"]):
        cases = selected["workload capacity payload".split()].drop_duplicates().sort_values(["capacity", "payload", "workload"])
        fig, ax = plt.subplots(figsize=(10, max(3, 0.9 * len(cases) + 1.4)))
        encodings = [(impl, name) for impl in implementations for name in RUNS]
        for index, (impl, name) in enumerate(encodings):
            offset = (index - (len(encodings) - 1) / 2) * (0.65 / len(encodings))
            for position, case in enumerate(cases.itertuples(index=False)):
                rows = selected.loc[selected["workload"].eq(case.workload) & selected["implementation"].eq(impl) & selected["run"].eq(name)]
                if rows.empty:
                    continue
                point = rows.iloc[0]
                median, low, high = (point[field] / scale for field in ["median", "low", "high"])
                color = COLORS.get(impl, "#777777")
                ax.errorbar(median, position + offset, xerr=[[median - low], [high - median]],
                            fmt=MARKERS.get(impl, "D"), color=color,
                            markerfacecolor="white" if name == "old" else color,
                            markeredgecolor="#1A1A1A", markeredgewidth=0.8,
                            markersize=9, capsize=5, capthick=2.3, elinewidth=2.3, zorder=3)
        labels = [f"{int(row.capacity)} slots / {int(row.payload)} B" for row in cases.itertuples(index=False)]
        ax.set(yticks=range(len(labels)), yticklabels=labels, xlim=limits,
               xlabel=unit, title=f"{title} · {int(producers)}P → {int(consumers)}C")
        if log_axis:
            ax.set_xscale("log")
        ax.invert_yaxis()
        ax.grid(axis="y", alpha=0.1)
        ax.legend(handles=legend_handles(implementations), loc="best", ncol=min(3, len(encodings)))
        fig.text(0.5, 0.015, "Marker: median  •  whiskers: minimum–maximum repeat", ha="center", fontsize=9)
        fig.tight_layout(rect=(0, 0.055, 1, 1))
        plt.show()


throughput_summary = summarize(compared_throughput, CASE_KEYS, "events_per_second")
horizontal_panels(throughput_summary, 1e6, "Million unique events / second", "Saturated throughput")
for field, scale, unit, title, log_axis in [
    ("cpu_cores", 1, "CPU cores used", "CPU at saturation", False),
    ("peak_rss_bytes", 1024**2, "Peak process RSS (MiB, log scale)", "RSS at saturation", True),
]:
    horizontal_panels(summarize(compared_throughput, CASE_KEYS, field), scale, unit, title, log_axis)

# Placement is diagnostic context rather than an automatic assertion of equal hardware.
placements = []
for name, frame in compared_throughput.items():
    placement = frame.groupby(CASE_KEYS)["cpus"].agg(lambda values: ", ".join(sorted(set(values.dropna().astype(str)))))
    placements.append(placement.rename(name))
placement_table = pd.concat(placements, axis=1).reset_index()
if not placement_table.empty:
    display(Markdown("**CPU placement for shared throughput cases:**"))
    display(placement_table)
    if placement_table["old"].ne(placement_table["new"]).any():
        display(Markdown("**Warning:** Some CPU placements differ; these are not controlled same-placement comparisons."))

## 3. Throughput change: new versus old

Percent change is `(new median / old median − 1) × 100`. Positive means higher throughput; negative means lower. These are ratios of independent medians, not paired-repeat estimates. Check the repeat ranges above before interpreting small changes.

In [ ]:
old_summary = throughput_summary.loc[throughput_summary["run"].eq("old")].drop(columns="run")
new_summary = throughput_summary.loc[throughput_summary["run"].eq("new")].drop(columns="run")
throughput_comparison = old_summary.merge(new_summary, on=CASE_KEYS, suffixes=("_old", "_new"), validate="one_to_one")
throughput_comparison["change_pct"] = (throughput_comparison["median_new"] / throughput_comparison["median_old"] - 1) * 100
if throughput_comparison.empty:
    display(Markdown("No shared throughput cases to compare."))
else:
    display(throughput_comparison[CASE_KEYS + ["median_old", "median_new", "repeats_old", "repeats_new", "change_pct"]].sort_values("change_pct"))
    for impl, selected in throughput_comparison.groupby("implementation"):
        selected = selected.sort_values("change_pct")
        fig, ax = plt.subplots(figsize=(10, max(3, 0.45 * len(selected) + 1.2)))
        ax.barh(selected["workload"], selected["change_pct"], color=COLORS.get(impl, "#777777"), alpha=0.85)
        ax.axvline(0, color="#666666", lw=1)
        ax.set(xlabel="Throughput change (%) · positive = higher", title=f"New versus old · {impl}")
        ax.invert_yaxis()
        ax.margins(x=0.15)
        fig.tight_layout()
        plt.show()

## 4. Latency at shared load fractions

For every complete repeat, take the **maximum consumer percentile**, then the median across repeats. Bands span repeat minima–maxima. These are summaries of percentiles, **not pooled event percentiles**. Implementation uses color/marker; run uses dashed/solid line.

The intersection here matches **load fraction**, not absolute offered rate. In a standalone run the fractions use Zigrupt's own saturated median; a competitive run uses the slowest selected implementation's median. The rate table quantifies the difference. Do **not** read these curves as equal-rate latency improvements/regressions when rates differ. The attainment plot diagnoses whether the scheduled rate was actually sustained.

In [ ]:
rate_summary = summarize(compared_latency, LATENCY_KEYS, "offered_rate")
old_rates = rate_summary.loc[rate_summary["run"].eq("old"), LATENCY_KEYS + ["median", "low", "high"]]
new_rates = rate_summary.loc[rate_summary["run"].eq("new"), LATENCY_KEYS + ["median", "low", "high"]]
rate_comparison = old_rates.merge(new_rates, on=LATENCY_KEYS, suffixes=("_old", "_new"), validate="one_to_one")
rate_comparison["new / old offered rate"] = rate_comparison["median_new"] / rate_comparison["median_old"]
rate_comparison["equal offered rate"] = (
    rate_comparison["low_old"].eq(rate_comparison["high_old"])
    & rate_comparison["low_new"].eq(rate_comparison["high_new"])
    & rate_comparison["median_old"].eq(rate_comparison["median_new"])
)
if rate_comparison.empty:
    display(Markdown("No shared complete latency cases/load fractions."))
else:
    display(rate_comparison[LATENCY_KEYS + ["median_old", "median_new", "new / old offered rate", "equal offered rate"]].rename(
        columns={"median_old": "old offered events/s", "median_new": "new offered events/s"}
    ))
    unequal = int((~rate_comparison["equal offered rate"]).sum())
    display(Markdown(f"**{unequal}/{len(rate_comparison)} shared latency points have different offered rates.** Compare the curves as run context, not equal-rate measurements."))


def workload_panels(
    frames: dict[str, pd.DataFrame], field: str, scale: float,
    ylabel: str, heading: str, log_axis: bool = False, reference: float | None = None,
) -> None:
    summary = summarize(frames, LATENCY_KEYS, field)
    if summary.empty:
        display(Markdown(f"No shared valid measurements for {heading.lower()}."))
        return
    implementations = sorted(summary["implementation"].unique())
    for (producers, consumers), selected in summary.groupby(["producers", "consumer_count"]):
        cases = selected[["workload", "capacity", "payload"]].drop_duplicates().sort_values(["capacity", "payload", "workload"])
        columns = min(2, len(cases))
        rows = math.ceil(len(cases) / columns)
        fig, axes = plt.subplots(rows, columns, figsize=(12, 3.6 * rows + 1), sharex=True, sharey=True, squeeze=False)
        fractions = sorted(selected["fraction"].unique() * 100)
        for ax, case in zip(axes.flat, cases.itertuples(index=False)):
            matching = selected.loc[selected["workload"].eq(case.workload)]
            for (impl, name), points in matching.groupby(["implementation", "run"]):
                points = points.sort_values("fraction")
                xs = points["fraction"].to_numpy() * 100
                medians, lows, highs = (points[key].to_numpy() / scale for key in ["median", "low", "high"])
                color = COLORS.get(impl, "#777777")
                ax.plot(xs, medians, color=color, marker=MARKERS.get(impl, "D"),
                        markerfacecolor="white" if name == "old" else color,
                        linestyle=LINESTYLES[name], lw=1.8, markersize=5)
                if (lows != highs).any():
                    ax.fill_between(xs, lows, highs, color=color, alpha=0.10 if name == "old" else 0.18)
            if reference is not None:
                ax.axhline(reference, color="#666666", linestyle=":", lw=1)
            ax.set(title=f"{int(case.capacity)} slots / {int(case.payload)} B", xticks=fractions)
            ax.set_xlim(min(fractions) - 3, max(fractions) + 3)
            if log_axis:
                ax.set_yscale("log")
        for ax in list(axes.flat)[len(cases):]:
            ax.set_visible(False)
        fig.suptitle(f"{heading} · {int(producers)}P → {int(consumers)}C", y=0.99)
        fig.supxlabel("Offered load / each run's reference saturated median (%)")
        fig.supylabel(ylabel)
        fig.legend(handles=legend_handles(implementations), loc="upper center",
                   bbox_to_anchor=(0.5, 0.95), ncol=min(4, 2 * len(implementations)), frameon=False)
        fig.tight_layout(rect=(0.03, 0.03, 1, 0.88))
        plt.show()


workload_panels(compared_latency, "p99_ns", 1_000, "Worst consumer p99 (µs, log scale)", "Tail latency · rates may differ", log_axis=True)
workload_panels(compared_latency, "attainment_pct", 1, "Achieved / offered rate (%)", "Rate attainment", reference=100)
for name, frame in compared_latency.items():
    rates = frame["attainment_pct"].dropna()
    if not rates.empty:
        print(f"{name}: {int(rates.lt(95).sum())}/{len(rates)} shared latency repeats achieved under 95% of offered rate.")

## 5. Workload detail: p50, p99, and p99.9

Set `DETAIL_WORKLOAD` to one of the shared workload keys. The default prefers a broadcast workload. Each curve is the median of worst-consumer summaries across complete repeats, with a minimum–maximum band. All three panels use microseconds on logarithmic axes; absolute offered rates can still differ as shown above. Zero-valued percentiles cannot appear on a log axis and are omitted.

In [ ]:
DETAIL_WORKLOAD: str | None = None  # e.g. "1p3c-2048-8b"
available = sorted(shared_latency["workload"].unique())
broadcast = sorted(shared_latency.loc[shared_latency["consumer_count"].gt(1), "workload"].unique())
selected_workload = DETAIL_WORKLOAD or (broadcast[0] if broadcast else (available[0] if available else None))
if selected_workload is None:
    display(Markdown("No shared complete latency workloads for a detail plot."))
elif selected_workload not in available:
    display(Markdown(f"`{selected_workload}` has no shared complete latency repeats. Available: {', '.join(available)}"))
else:
    selected = {name: frame.loc[frame["workload"].eq(selected_workload)] for name, frame in compared_latency.items()}
    implementations = sorted(shared_latency.loc[shared_latency["workload"].eq(selected_workload), "implementation"].unique())
    fig, axes = plt.subplots(1, 3, figsize=(14, 5.8), sharex=True)
    for ax, field, title in zip(axes, PERCENTILES, ["p50", "p99", "p99.9"]):
        summary = summarize(selected, LATENCY_KEYS, field)
        for (impl, name), points in summary.groupby(["implementation", "run"]):
            points = points.sort_values("fraction")
            xs = points["fraction"].to_numpy() * 100
            median, low, high = (points[key].to_numpy() / 1_000 for key in ["median", "low", "high"])
            color = COLORS.get(impl, "#777777")
            ax.plot(xs, median, color=color, marker=MARKERS.get(impl, "D"),
                    markerfacecolor="white" if name == "old" else color,
                    linestyle=LINESTYLES[name], lw=1.8)
            ax.fill_between(xs, low, high, color=color, alpha=0.10 if name == "old" else 0.18)
        ax.set(title=title, xlabel="Offered load (%)", ylabel="Worst consumer latency (µs, log scale)")
        ax.set_yscale("log")
    fig.suptitle(f"Worst-consumer latency · {selected_workload} · rates may differ", y=0.98)
    fig.legend(handles=legend_handles(implementations), loc="upper center", bbox_to_anchor=(0.5, 0.93),
               ncol=min(4, 2 * len(implementations)), frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.82))
    plt.show()